# 📊 GoldMind - Exploratory Data Analysis (EDA)
Comprehensive data quality checks, price trends, return distributions, session volatility patterns, and autocorrelation/volatility clustering for XAU/USD (Gold).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CSV_PATH = "data1/XAU_1m_dataX.csv"
OUT_DIR = "eda_output"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"EDA setup ready. Output directory: {OUT_DIR}")


In [ ]:
# ---------- 1. Load Data & Resample to Hourly ----------
print(f"Loading {CSV_PATH} ...")
raw = pd.read_csv(CSV_PATH, parse_dates=["Date"])
raw = raw.sort_values("Date").reset_index(drop=True)

print(f"Loaded {len(raw):,} 1-minute bars from {raw['Date'].min()} to {raw['Date'].max()}")

# Resample to 1-Hour bars
df_1h = (
    raw.set_index("Date")
    .resample("1h")
    .agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"})
    .dropna()
)
print(f"Resampled to {len(df_1h):,} 1-hour bars.")


In [ ]:
# ---------- 2. Data Quality & Integrity Checks ----------
print("=== Data Quality Checks ===")
print("Missing values in raw data:\n", raw.isna().sum())
print("Duplicate timestamps:", raw["Date"].duplicated().sum())

# OHLC consistency check
broken_ohlc = raw[
    (raw["High"] < raw["Low"])
    | (raw["Open"] > raw["High"]) | (raw["Open"] < raw["Low"])
    | (raw["Close"] > raw["High"]) | (raw["Close"] < raw["Low"])
]
print(f"Broken OHLC bars: {len(broken_ohlc)}")

# Non-positive prices or zero volume
print("Non-positive prices:", (raw[["Open", "High", "Low", "Close"]] <= 0).sum().sum())
print("Zero-volume bars:", (raw["Volume"] == 0).sum())

# Weekend and exchange holiday gaps (> 6 hours)
gaps = raw["Date"].diff()
big_gaps = gaps[gaps > pd.Timedelta(hours=6)]
print(f"Weekend / Market gaps (>6h): {len(big_gaps)}  |  Largest gap: {gaps.max()}")


In [ ]:
# ---------- 3. Plot Price & Volume Trends ----------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(df_1h.index, df_1h["Close"], color="#d4af37", lw=1.2, label="XAU/USD Close")
ax1.set_title("XAU/USD (Gold) Hourly Price Trend", fontsize=14, fontweight='bold')
ax1.set_ylabel("Price (USD)")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper left")

ax2.bar(df_1h.index, df_1h["Volume"], color="#4682b4", alpha=0.6, width=0.03, label="Volume")
ax2.set_title("Trading Volume", fontsize=11)
ax2.set_ylabel("Volume")
ax2.set_xlabel("Date")
ax2.grid(True, alpha=0.3)
ax2.legend(loc="upper left")

plt.tight_layout()
fig_path = os.path.join(OUT_DIR, "price_volume_trend.png")
plt.savefig(fig_path, dpi=200)
plt.show()
print(f"Saved price & volume plot to {fig_path}")


In [ ]:
# ---------- 4. Return Distribution & Outlier Analysis ----------
df_1h["ret"] = df_1h["Close"].pct_change()
clean_ret = df_1h["ret"].dropna()

mean_ret = clean_ret.mean()
std_ret = clean_ret.std()
skew_ret = clean_ret.skew()
kurt_ret = clean_ret.kurtosis()
var_95 = clean_ret.quantile(0.05)

print("=== 1-Hour Return Statistics ===")
print(f"Mean Return      : {mean_ret:.6f} ({mean_ret*100:.4f}%)")
print(f"Hourly Vol (Std) : {std_ret:.6f} ({std_ret*100:.4f}%)")
print(f"Skewness         : {skew_ret:.4f}")
print(f"Excess Kurtosis  : {kurt_ret:.4f} (Fat tails / Leptokurtic)")
print(f"VaR 95% (1-Hour) : {var_95*100:.3f}%")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(clean_ret, bins=100, density=True, alpha=0.65, color="#2b5c8f", label="Empirical Returns")

# Overlay normal distribution
x = np.linspace(clean_ret.min(), clean_ret.max(), 500)
norm_pdf = (1 / (std_ret * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean_ret) / std_ret) ** 2)
ax.plot(x, norm_pdf, 'r--', lw=1.5, label="Normal Distribution")

ax.set_title(f"1-Hour Return Distribution (Kurtosis = {kurt_ret:.2f})", fontsize=12, fontweight='bold')
ax.set_xlabel("Return")
ax.set_ylabel("Density")
ax.set_xlim(-0.02, 0.02)
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
dist_path = os.path.join(OUT_DIR, "return_distribution.png")
plt.savefig(dist_path, dpi=200)
plt.show()
print(f"Saved return distribution plot to {dist_path}")


In [ ]:
# ---------- 5. Session & Hourly Volatility Patterns ----------
df_1h["Hour"] = df_1h.index.hour
hourly_vol = df_1h.groupby("Hour")["ret"].std() * 100

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(hourly_vol.index, hourly_vol.values, color="#e67e22", alpha=0.8, edgecolor="#b95e09")

# Highlight London & NY overlap
ax.axvspan(7, 16, color="blue", alpha=0.08, label="London Session (07-16 UTC)")
ax.axvspan(12, 20, color="green", alpha=0.08, label="NY Session (12-20 UTC)")

ax.set_title("Hourly Volatility Profile (Average % Standard Deviation by Hour UTC)", fontsize=12, fontweight='bold')
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Volatility (%)")
ax.set_xticks(range(24))
ax.grid(True, alpha=0.3, axis="y")
ax.legend(loc="upper left")

plt.tight_layout()
vol_path = os.path.join(OUT_DIR, "hourly_volatility.png")
plt.savefig(vol_path, dpi=200)
plt.show()
print(f"Saved hourly volatility profile to {vol_path}")


In [ ]:
# ---------- 6. Autocorrelation & Volatility Clustering (ARCH Effect) ----------
lags = range(1, 25)
acf_ret = [clean_ret.autocorr(lag=l) for l in lags]
acf_vol = [clean_ret.abs().autocorr(lag=l) for l in lags]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(lags, acf_ret, color="#34495e", alpha=0.8)
ax1.axhline(0, color="black", lw=0.8)
ax1.axhline(1.96 / np.sqrt(len(clean_ret)), color="red", linestyle="--", alpha=0.6, label="95% CI")
ax1.axhline(-1.96 / np.sqrt(len(clean_ret)), color="red", linestyle="--", alpha=0.6)
ax1.set_title("ACF of Raw Returns (Price Random Walk Check)", fontsize=11, fontweight='bold')
ax1.set_xlabel("Lag (Hours)")
ax1.set_ylabel("Autocorrelation")
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.bar(lags, acf_vol, color="#e74c3c", alpha=0.8)
ax2.axhline(0, color="black", lw=0.8)
ax2.axhline(1.96 / np.sqrt(len(clean_ret)), color="red", linestyle="--", alpha=0.6, label="95% CI")
ax2.set_title("ACF of Absolute Returns (Volatility Clustering / ARCH)", fontsize=11, fontweight='bold')
ax2.set_xlabel("Lag (Hours)")
ax2.set_ylabel("Autocorrelation of |Return|")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
acf_path = os.path.join(OUT_DIR, "autocorrelation_clustering.png")
plt.savefig(acf_path, dpi=200)
plt.show()
print(f"Saved autocorrelation plot to {acf_path}")


In [ ]:
# ---------- 7. Yearly Summary Table ----------
df_1h["Year"] = df_1h.index.year
yearly = df_1h.groupby("Year").agg(
    Open=("Open", "first"),
    Close=("Close", "last"),
    High=("High", "max"),
    Low=("Low", "min"),
    Bars=("Close", "count"),
    Avg_Volume=("Volume", "mean"),
    Annual_Return=("Close", lambda s: (s.iloc[-1] / s.iloc[0]) - 1.0)
)
yearly["Annual_Return_Pct"] = yearly["Annual_Return"].map(lambda x: f"{x*100:+.2f}%")
print("\n=== Yearly Summary ===")
print(yearly[["Open", "High", "Low", "Close", "Bars", "Annual_Return_Pct"]])

yearly_path = os.path.join(OUT_DIR, "yearly_summary.csv")
yearly.to_csv(yearly_path)
print(f"Saved yearly summary to {yearly_path}")
